# MedTrack_DV – Milestone 1
## Notebook 03: Data Normalization
**Purpose:** Standardize IDs, department names, date formats, and categorical values across all datasets so they can be linked properly in Tableau

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW = '../data/raw/'

# Load all cleaned data (re-running cleaning steps)
hmis_sheets      = pd.read_excel(RAW + 'Hospital_Management_System.xlsx', sheet_name=None)
df_bed_records   = hmis_sheets['BedRecords'].copy()
df_department    = hmis_sheets['Department'].copy()
df_doctor        = hmis_sheets['Doctor'].copy()
df_nurse         = hmis_sheets['Nurse'].copy()
df_patients      = hmis_sheets['Patients'].copy()
df_ward          = hmis_sheets['Ward'].copy()
df_bed           = hmis_sheets['Bed'].copy()
df_staff_shift   = hmis_sheets['StaffShift'].copy()

df_beds_patients = pd.read_csv(RAW + 'beds_patients.csv')
df_beds_services = pd.read_csv(RAW + 'beds_services_weekly.csv')
df_readmission   = pd.read_csv(RAW + 'readmission_admission_data.csv')
df_healthcare    = pd.read_csv(RAW + 'healthcare_dataset.csv')

# Standardize column names
for df in [df_bed_records, df_department, df_doctor, df_nurse,
           df_patients, df_ward, df_bed, df_staff_shift,
           df_beds_patients, df_beds_services, df_readmission, df_healthcare]:
    df.columns = [c.strip().lower().replace(' ', '_').replace('.', '') for c in df.columns]

print('All data loaded for normalization!')

## Step 1: Standardize Department Names

In [ ]:
# Standard department name mapping
dept_name_map = {
    'Er': 'Emergency Department',
    'E.R': 'Emergency Department',
    'Emergency Room': 'Emergency Department',
    'Emergency Dept': 'Emergency Department',
    'Icu': 'ICU',
    'Intensive Care': 'ICU',
    'Intensive Care Unit': 'ICU',
    'Gen Ward': 'General Ward',
    'General': 'General Ward',
    'Cardio': 'Cardiology',
    'Ortho': 'Orthopedics',
    'Paeds': 'Pediatrics',
    'Paediatrics': 'Pediatrics',
    'Gynaecology': 'Gynecology',
    'Obs & Gynae': 'Obstetrics & Gynecology',
    'Obs And Gynae': 'Obstetrics & Gynecology'
}

def standardize_dept(name):
    if pd.isna(name):
        return 'Unknown'
    name = str(name).strip().title()
    return dept_name_map.get(name, name)

# Apply to HMIS department
df_department['dept_name'] = df_department['dept_name'].apply(standardize_dept)

# Apply to beds services
df_beds_services['service'] = df_beds_services['service'].apply(standardize_dept)
df_beds_patients['service'] = df_beds_patients['service'].apply(standardize_dept)

print('Department names standardized!')
print('\nDepartments in HMIS:')
print(df_department['dept_name'].tolist())

## Step 2: Standardize Date Formats

In [ ]:
def parse_dates(df, date_cols):
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

# HMIS BedRecords
df_bed_records = parse_dates(df_bed_records, ['admission_date', 'discharge_date'])
df_bed_records['length_of_stay_days'] = (df_bed_records['discharge_date'] - df_bed_records['admission_date']).dt.days
df_bed_records['admission_year']   = df_bed_records['admission_date'].dt.year
df_bed_records['admission_month']  = df_bed_records['admission_date'].dt.month
df_bed_records['admission_month_name'] = df_bed_records['admission_date'].dt.month_name()
df_bed_records['admission_quarter'] = df_bed_records['admission_date'].dt.quarter
df_bed_records['day_of_week']      = df_bed_records['admission_date'].dt.day_name()

# Beds Patients
df_beds_patients = parse_dates(df_beds_patients, ['arrival_date', 'departure_date'])
df_beds_patients['length_of_stay'] = (df_beds_patients['departure_date'] - df_beds_patients['arrival_date']).dt.days

# Readmission
df_readmission = parse_dates(df_readmission, ['d_o_a', 'd_o_d'])

# Healthcare
df_healthcare = parse_dates(df_healthcare, ['date_of_admission', 'discharge_date'])
df_healthcare['length_of_stay_days'] = (df_healthcare['discharge_date'] - df_healthcare['date_of_admission']).dt.days
df_healthcare['admission_year']  = df_healthcare['date_of_admission'].dt.year
df_healthcare['admission_month'] = df_healthcare['date_of_admission'].dt.month_name()
df_healthcare['admission_quarter'] = df_healthcare['date_of_admission'].dt.quarter

print('Date formats standardized!')

## Step 3: Standardize Gender Values

In [ ]:
def standardize_gender(val):
    if pd.isna(val):
        return 'Unknown'
    val = str(val).strip().upper()
    if val in ['M', 'MALE']:
        return 'Male'
    elif val in ['F', 'FEMALE']:
        return 'Female'
    return 'Unknown'

df_patients['gender']     = df_patients['gender'].apply(standardize_gender)
df_doctor['gender']       = df_doctor['gender'].apply(standardize_gender)
df_nurse['gender']        = df_nurse['gender'].apply(standardize_gender)
df_readmission['gender']  = df_readmission['gender'].apply(standardize_gender)
df_healthcare['gender']   = df_healthcare['gender'].apply(standardize_gender)

print('Gender values standardized!')
print('HMIS Patients gender:', df_patients['gender'].value_counts().to_dict())

## Step 4: Standardize Admission Types

In [ ]:
def standardize_admission_type(val):
    if pd.isna(val):
        return 'Unknown'
    val = str(val).strip().title()
    mapping = {
        'Emergency': 'Emergency',
        'Opd': 'OPD',
        'Outpatient': 'OPD',
        'Elective': 'Elective',
        'Urgent': 'Urgent',
        'Planned': 'Elective'
    }
    return mapping.get(val, val)

if 'type_of_admission-emergency/opd' in df_readmission.columns:
    df_readmission['admission_type'] = df_readmission['type_of_admission-emergency/opd'].apply(standardize_admission_type)
elif 'admission_type' in df_readmission.columns:
    df_readmission['admission_type'] = df_readmission['admission_type'].apply(standardize_admission_type)

df_healthcare['admission_type'] = df_healthcare['admission_type'].apply(standardize_admission_type)

print('Admission types standardized!')

## Step 5: Create Surrogate IDs for Linkage

In [ ]:
# Add hospital_id to all datasets (since HMIS doesn't have a hospital column)
df_bed_records['hospital_id']   = 'H001'
df_bed_records['hospital_name'] = 'MedTrack General Hospital'

df_beds_patients['hospital_id']   = 'H001'
df_beds_patients['hospital_name'] = 'MedTrack General Hospital'

df_beds_services['hospital_id']   = 'H001'
df_beds_services['hospital_name'] = 'MedTrack General Hospital'

# Ensure admission_id exists in bed_records
if 'admission_id' not in df_bed_records.columns:
    df_bed_records = df_bed_records.reset_index(drop=True)
    df_bed_records['admission_id'] = ['ADM' + str(i).zfill(5) for i in range(1, len(df_bed_records)+1)]

print('Surrogate IDs created!')
print(df_bed_records[['admission_id','hospital_id','hospital_name','patient_id']].head(3))

## Step 6: Save Normalized Datasets

In [ ]:
PROCESSED = '../data/processed/'

# Save normalized tables
df_patients.to_csv(PROCESSED + 'normalized_patients.csv', index=False)
df_bed_records.to_csv(PROCESSED + 'normalized_bed_records.csv', index=False)
df_department.to_csv(PROCESSED + 'normalized_departments.csv', index=False)
df_doctor.to_csv(PROCESSED + 'normalized_doctors.csv', index=False)
df_nurse.to_csv(PROCESSED + 'normalized_nurses.csv', index=False)
df_ward.to_csv(PROCESSED + 'normalized_wards.csv', index=False)
df_bed.to_csv(PROCESSED + 'normalized_beds.csv', index=False)
df_staff_shift.to_csv(PROCESSED + 'normalized_staff_shifts.csv', index=False)
df_beds_patients.to_csv(PROCESSED + 'normalized_beds_patients.csv', index=False)
df_beds_services.to_csv(PROCESSED + 'normalized_beds_services.csv', index=False)
df_readmission.to_csv(PROCESSED + 'normalized_readmission.csv', index=False)
df_healthcare.to_csv(PROCESSED + 'normalized_healthcare.csv', index=False)

print('All normalized datasets saved to processed folder!')

## ✅ Normalization Complete!
Proceed to **04_data_validation.ipynb**